<a href="https://colab.research.google.com/github/pedrosouzag/edicao-imagens-ia/blob/main/semana5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# instalando as bibliotecas
!pip install datasets pandas pillow -q

from datasets import load_dataset
from PIL import Image
import pandas as pd
import os
import random

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
dataset = load_dataset("detection-datasets/coco", split="train")
print(f"Total de imagens no dataset: {len(dataset)}")


In [ ]:
categoria_names = dataset.features["objects"]["category"].feature.names
print(categoria_names[:20])

['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow']


In [ ]:
# categorias de imagens
categorias_desejadas = ["person", "cup", "chair", "dining table", "car", "backpack", "dog", "cat"]

In [ ]:
# criar pasta para salvar imagens
os.makedirs("/content/drive/MyDrive/dataset/images", exist_ok=True)

In [ ]:
import random

random.seed(42)

indices = list(range(len(dataset)))
random.shuffle(indices)

metadata = []

contador = 0
max_imagens = 1000

for idx in indices:

    item = dataset[idx]

    labels = item["objects"]["category"]
    areas = item["objects"]["area"]

    nomes = []
    for i in labels:
        nome = dataset.features["objects"]["category"].feature.names[i]
        nomes.append(nome)

    # Verifica se existe alguma categoria desejada COM tamanho aceitavel
    encontrou = False
    for nome, area in zip(nomes, areas):
      if nome in categorias_desejadas and area > 35000:
        encontrou = True
        categoria_encontrada = nome
        break

    if not encontrou:
        continue

    imagem = item["image"]

    largura, altura = imagem.size

    # Ignora imagens pequenas
    if largura < 400 or altura < 400:
        continue

    # Redimensiona para 512x512
    imagem = imagem.resize((512,512))

    nome = f"img_{contador:04d}.png"

    caminho = os.path.join("/content/drive/MyDrive/dataset/images", nome)

    imagem.save(caminho)

    metadata.append({"arquivo": nome, "categorias": ", ".join(nomes), "largura": 512, "altura": 512})

    contador += 1

    if contador >= max_imagens:
        break

print(f"{contador} imagens salvas.")

In [ ]:


# random.seed(42)

# indices = list(range(len(dataset)))
# random.shuffle(indices)

# categorias_desejadas = [
#     "car",
#     "chair",
#     "dining table",
#     "backpack"
# ]

# LIMITE = 30

# contador_categoria = {cat: 0 for cat in categorias_desejadas}

# base = "/content/drive/MyDrive/dataset"

# for categoria in categorias_desejadas:
#     pasta = os.path.join(base, categoria.replace(" ", "_"))
#     os.makedirs(pasta, exist_ok=True)

# metadata = []

# for idx in indices:

#     if all(v >= LIMITE for v in contador_categoria.values()):
#         break

#     item = dataset[idx]

#     labels = item["objects"]["category"]
#     areas = item["objects"]["area"]

#     nomes = [dataset.features["objects"]["category"].feature.names[i] for i in labels]

#     categoria_escolhida = None

#     for nome, area in zip(nomes, areas):
#         if nome in categorias_desejadas and area > 35000 and contador_categoria[nome] < LIMITE:
#             categoria_escolhida = nome
#             break

#     if categoria_escolhida is None:
#         continue

#     imagem = item["image"]

#     largura, altura = imagem.size

#     if largura < 400 or altura < 400:
#         continue

#     imagem = imagem.resize((512, 512))

#     contador_categoria[categoria_escolhida] += 1
#     numero = contador_categoria[categoria_escolhida]

#     pasta = os.path.join(base, categoria_escolhida.replace(" ", "_"))
#     arquivo = f"{categoria_escolhida.replace(' ','_')}_{numero:03d}.png"
#     caminho = os.path.join(pasta, arquivo)

#     imagem.save(caminho)

#     metadata.append({"arquivo": arquivo, "categoria_principal": categoria_escolhida, "todas_as_categorias": ", ".join(nomes), "largura": 512, "altura": 512})

# df = pd.DataFrame(metadata)
# df.to_csv(os.path.join(base, "metadata2.csv"), index=False)

# print("\nQuantidade por categoria:")

# for categoria, quantidade in contador_categoria.items():
#     print(f"{categoria}: {quantidade}")

# print("\nmetadata2.csv criado com sucesso.")


Quantidade por categoria:
car: 30
chair: 30
dining table: 30
backpack: 30

metadata2.csv criado com sucesso.


In [ ]:
import pandas as pd

# transforma a lista metadata numa tabela
df = pd.DataFrame(metadata)

# salva a tabela como CSV no Drive
df.to_csv("/content/drive/MyDrive/dataset/metadata.csv", index=False)

print(f"Metadados salvos: {len(df)} linhas")